In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
from pathlib import Path
from prophet import Prophet

import warnings
import prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_plotly, plot_components_plotly

import matplotlib.pyplot as plt


import pandas as pd
import numpy as np
import seaborn as sns
from loguru import logger
import sys

import logging
logging.getLogger("cmdstanpy").disabled = True


warnings.filterwarnings("ignore", category=FutureWarning)
logger.remove()
logger.add(sys.stderr, level="INFO", format="{time:HH:mm:ss} | {level:<7} | {message}")
logger.info("FE Avance 2 — EpiForecast-MX inicializado")

In [ ]:
# =============================================================================
# CONSTANTES Y CONFIGURACIÓN
# =============================================================================

# --- Rutas -------------------------------------------------------------------
# Dataset ya preparado para hacer merge con INEGI (sale del pipeline de limpieza/transform)
DATA_PATH = Path("../data/processed/data_inegi_General.csv")

# --- Paleta IMSS institucional (Guía cromática oficial) ----------------------
IMSS_COLORS = {
    "neutral_black":  "#231F20",  # PANTONE Neutral Black C
    "burgundy":       "#9B2242",  # PANTONE 7420 C
    "dark_burgundy":  "#6F1D46",  # PANTONE 7421 C
    "cool_gray":      "#97999B",  # PANTONE Cool Gray C
    "teal":           "#00524E",  # PANTONE IMSS 561 C
    "dark_teal":      "#173F35",  # PANTONE 627 C
    "cream":          "#E8D5B5",  # PANTONE 7402 C
    "gold":           "#B58500",  # PANTONE 1255 C
}

# Paleta secuencial para gráficos
PALETTE_MAIN = [
    IMSS_COLORS["teal"],
    IMSS_COLORS["burgundy"],
    IMSS_COLORS["gold"],
    IMSS_COLORS["dark_teal"],
    IMSS_COLORS["dark_burgundy"],
    IMSS_COLORS["cool_gray"],
    IMSS_COLORS["neutral_black"],
    IMSS_COLORS["cream"],
]

PALETTE_PADECIMIENTO = {
    "Depresión":  IMSS_COLORS["burgundy"],
    "Parkinson":  IMSS_COLORS["teal"],
    "Alzheimer":  IMSS_COLORS["gold"],
}

PALETTE_SEXO = {
    "Hombres": IMSS_COLORS["teal"],
    "Mujeres": IMSS_COLORS["burgundy"],
}

# --- Estilo global de matplotlib ---------------------------------------------
plt.rcParams.update({
    "figure.facecolor":    "white",
    "axes.facecolor":      "white",
    "axes.edgecolor":      IMSS_COLORS["cool_gray"],
    "axes.labelcolor":     IMSS_COLORS["neutral_black"],
    "text.color":          IMSS_COLORS["neutral_black"],
    "xtick.color":         IMSS_COLORS["neutral_black"],
    "ytick.color":         IMSS_COLORS["neutral_black"],
    "axes.grid":           True,
    "grid.alpha":          0.3,
    "grid.color":          IMSS_COLORS["cool_gray"],
    "font.family":         "sans-serif",
    "font.size":           11,
    "axes.titlesize":      13,
    "axes.titleweight":    "bold",
    "figure.titlesize":    15,
    "figure.titleweight":  "bold",
    "figure.dpi":          120,
    "savefig.dpi":         150,
    "savefig.bbox":        "tight",
})

logger.success(f"Configuración cargada | Paleta IMSS: {len(IMSS_COLORS)} colores")

In [ ]:
df_datos = pd.read_csv(DATA_PATH)
df_datos['Fecha'] = pd.to_datetime(df_datos['Fecha'])
df_datos.info()

In [ ]:
serie  = (df_datos
                .groupby(["Fecha","Entidad"])[["incrementos_hombres", "incrementos_mujeres"]]
                .sum()
                .reset_index()
                .rename(columns={'Fecha':'ds'})
            )

serie["y"] = serie["incrementos_hombres"] + serie["incrementos_mujeres"]
serie = serie.sort_values('ds')
regiones = sorted(serie['Entidad'].unique())
serie.head(5)

In [ ]:

def calcular_mape(y_true, y_pred):
    den = y_true.replace(0, np.nan)
    mape_series = np.abs((y_true - y_pred) / den) * 100
    return mape_series.mean()

def predict(modelo):
    future = modelo.make_future_dataframe(periods = 84, freq='w')
    forecast = modelo.predict(future)
    return forecast   

def graficar(modelo,forecast,region):
    fig = modelo.plot(forecast)

    ax = fig.gca()
    ax.set_title(f"Pronóstico para {region}")
    ax.grid(True, linestyle='--', alpha=0.6)

    lineas = ax.get_lines()
    
    if lineas:
        lineas[0].set_color(IMSS_COLORS["dark_teal"])
        lineas[1].set_color(IMSS_COLORS["burgundy"]) 
        lineas[0].set_linewidth(2)

    for col in ax.collections:
        col.set_facecolor(IMSS_COLORS["cream"])
        col.set_edgecolor(IMSS_COLORS["gold"])

    #plt.savefig(f"graficos/pronostico_{region}.png", dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    fig2 = modelo.plot_components(forecast)
    for ax in fig2.get_axes():
        ax.set_title(f"componentes para {region}")
        ax.grid(True, linestyle='--', alpha=0.6)   
        ax.set_xlabel(None)
        ax.set_facecolor("white")

    # Cambiar color de las líneas en cada subplot → guinda
        for line in ax.get_lines():
            line.set_color(IMSS_COLORS["burgundy"])
            line.set_linewidth(2)

        for col in ax.collections:
            col.set_facecolor(IMSS_COLORS["cream"])
            col.set_edgecolor(IMSS_COLORS["gold"])

    #fig2.savefig(f"graficos/componentes_{region}.png", dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig2)


def evaluar_region(region, serie):
    modelo = Prophet()
    serie_prophet = serie.loc[serie['Entidad'] == region, ['ds', 'y']].copy()
    modelo.fit(serie_prophet)

    forecast = predict(modelo)

    cv_prophet = cross_validation(
        modelo,
        initial='730 days',
        period='56 days',
        horizon='168 days'
    )

    df_pm = performance_metrics(cv_prophet)

    metricas_disponibles = [c for c in ['rmse', 'mae', 'mdape'] if c in df_pm.columns]
    resumen_base = df_pm[metricas_disponibles].mean(numeric_only=True).to_dict()

    resumen_base['mape'] = calcular_mape(cv_prophet['y'], cv_prophet['yhat'])

    orden = ['rmse', 'mae', 'mape', 'mdape']
    resumen_ordenado = [resumen_base.get(k, np.nan) for k in orden]
    resumen = [region] + resumen_ordenado

    region, *metricas = resumen
    metricas_fmt = [f"{float(m):.2f}" for m in metricas]
    logger.success(f"Métricas para Entidad: {region}")
    logger.success("RMSE: {} | MAE: {} | MAPE: {} | MDAPE: {}".format(*metricas_fmt))

    graficar(modelo,forecast,region)

    return forecast, resumen

In [ ]:

resultados = []
forecasts = []

for region in regiones:
    logger.info(f'Entrenando región: {region}')
    forecast, resultado = evaluar_region(region,serie)
    forecasts.append(forecast)
    resultados.append(resultado)


In [ ]:
df_resultados = pd.DataFrame(resultados, columns=['región', 'rmse', 'mae', 'mape', 'mdape'])
print(df_resultados.sort_values('región'))


In [ ]:
data = [['Aguascalientes', 13.411336024850378,
         {'seasonality_mode': 'multiplicative', 
          'changepoint_prior_scale': 1.0, 
          'seasonality_prior_scale': 0.5}]]


df = pd.DataFrame(data, columns=['estado', 'valor', 'params'])
params_expandidos = pd.json_normalize(df['params'])
df = pd.concat([df.drop(columns=['params']), params_expandidos], axis=1)
print(df)
#df_final.to_csv('prueba.xlsx')
